# Assignment 2

Deadline: 25.03.2025, 12:00 CET
- Gallo Alessandro , 25-732-140 , alessandro.gallo2@uzh.ch
- Maruccio Anna , 25-742-800 , anna.maruccio@uzh.ch
- Perbellini Cesare, 25-741-257, cesare.perbellini@uzh.ch
- Venturi Matilde , 25-741-059 , matilde.venturi@uzh.ch

## 1. Linearization of Turnover

**(15 points)**

Turnover constraints are used to limit the amount of change in portfolio weights between periods, helping to manage transaction costs and maintain portfolio stability.

Your task is to implement a method `linearize_turnover_constraint` for the class `QuadraticProgram`, which modifies the quadratic programming problem to incorporate a linearized turnover constraint. This will involve updating the objective function coefficients, equality and inequality constraints, as well as the lower and upper bounds of the problem. 

Additionally, complete the example provided below to demonstrate that your method functions correctly.

In class, we discussed a solution that involved augmenting the dimensionality by a factor of three. Here, you are asked to implement an alternative method that involves a two-fold increase in dimensions. If you are unable to implement the two-fold method, you may proceed with the three-fold approach.

### Function Parameters:
- `x_init` (np.ndarray): The initial portfolio weights.
- `to_budget` (float, optional): The maximum allowable turnover. Defaults to `float('inf')`.

### Steps for Function Implementation:

As discussed in the lecture, introduce auxiliary variables and augment the matrices/vectors used for optimization.

- **Objective Function Coefficients**:  
  Pad the existing objective function coefficients (`P` and `q`) to accommodate the new variables introduced by the turnover constraint.  
  *Note*: "Padding" refers to adding extra elements (typically zeros) to an array or matrix to increase its size to a desired shape.

- **Equality Constraints**:  
  Pad the existing equality constraint matrix (`A`) to account for the new variables.

- **Inequality Constraints**:  
  Pad the existing inequality constraint matrix ('G') and vector ('h') and further add a new inequality constraint row to incorporate the turnover constraint.  

- **Lower and Upper Bounds**:  
  Pad the existing lower (`lb`) and upper (`ub`) bounds to accommodate the new variables.

- **Update Problem Data**:  
  Overwrite the original problem data in the `QuadraticProgram` class with the updated matrices and vectors to include the linearized turnover constraint.

In [8]:
# Import standard libraries
import types
import os
import sys

# Import third-party libraries
import numpy as np
import pandas as pd

# Import local modules
project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)
from estimation.covariance import Covariance
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.quadratic_program import QuadraticProgram
from helper_functions import load_data_msci

### Linearization of the Turnover Constraint

The portfolio turnover constraint is

$$
\sum_{i=1}^{n} |w_i - w_i^{0}| \le \tau,
$$

where $w$ denotes the portfolio weights, $w^0$ the initial portfolio, and $\tau$ the turnover budget.

Since the absolute value is not linear, the constraint must be reformulated before it can be included in a quadratic program.

### 2-Fold Formulation

In the 2-fold formulation, we introduce one auxiliary vector $t \in \mathbb{R}^n$ with $t_i \ge 0$, representing an upper bound on the trade magnitude of asset $i$. We impose

$$
t_i \ge |w_i - w_i^0|,
$$

which is equivalent to the two linear inequalities

$$
w_i - w_i^0 \le t_i,
\qquad
-(w_i - w_i^0) \le t_i.
$$

The turnover budget is then written as

$$
\sum_{i=1}^{n} t_i \le \tau.
$$

The augmented decision vector becomes

$$
x =
\begin{bmatrix}
w \\
t
\end{bmatrix}
\in \mathbb{R}^{2n},
$$

which explains the name **2-fold augmentation**.

Since the auxiliary variables $t$ do not affect the objective function, the matrices are extended as

$$
P_{\text{new}} =
\begin{bmatrix}
P & 0 \\
0 & 0
\end{bmatrix},
\qquad
q_{\text{new}} =
\begin{bmatrix}
q \\
0
\end{bmatrix}.
$$

The equality constraints remain unchanged apart from zero padding:

$$
A_{\text{new}} =
\begin{bmatrix}
A & 0
\end{bmatrix}.
$$

The original inequality constraints become

$$
\begin{bmatrix}
G & 0
\end{bmatrix},
$$

and the turnover constraints are added as

$$
w - t \le w^0,
\qquad
-w - t \le -w^0,
\qquad
\mathbf{1}^\top t \le \tau.
$$

The bounds become

$$
lb_{\text{new}} =
\begin{bmatrix}
lb \\
0
\end{bmatrix},
\qquad
ub_{\text{new}} =
\begin{bmatrix}
ub \\
+\infty
\end{bmatrix}.
$$

Thus, the original bounds on $w$ remain unchanged, while $t_i \ge 0$ for all $i$.

### 3-Fold Formulation

In the 3-fold formulation, turnover is linearized by introducing two nonnegative vectors

$$
w^+ \in \mathbb{R}^n, \qquad w^- \in \mathbb{R}^n,
$$

where $w_i^+$ and $w_i^-$ represent, respectively, purchases and sales of asset $i$.

The portfolio is written as

$$
w = w^0 + w^+ - w^-,
\qquad
w^+ \ge 0,\quad w^- \ge 0.
$$

The turnover budget becomes

$$
\sum_{i=1}^n (w_i^+ + w_i^-) \le \tau.
$$

The augmented decision vector is then

$$
x =
\begin{bmatrix}
w \\
w^+ \\
w^-
\end{bmatrix}
\in \mathbb{R}^{3n},
$$

which explains the name **3-fold augmentation**.

This formulation makes purchases and sales explicit, but it requires more auxiliary variables than the 2-fold formulation.

### Equivalence of the 2-Fold and 3-Fold Formulations

Although the two formulations use different auxiliary variables, they define the same feasible set for the portfolio weights $w$.

If $(w,w^+,w^-)$ is feasible in the 3-fold formulation, define

$$
t = w^+ + w^-.
$$

Since

$$
w - w^0 = w^+ - w^-,
$$

it follows componentwise that

$$
|w_i - w_i^0| = |w_i^+ - w_i^-| \le w_i^+ + w_i^- = t_i.
$$

Hence,

$$
t_i \ge |w_i - w_i^0|,
\qquad
\sum_{i=1}^n t_i = \sum_{i=1}^n (w_i^+ + w_i^-) \le \tau.
$$

So every feasible 3-fold solution induces a feasible 2-fold solution.

Conversely, if $(w,t)$ is feasible in the 2-fold formulation, define

$$
w^+ = \frac{t + (w-w^0)}{2},
\qquad
w^- = \frac{t - (w-w^0)}{2}.
$$

Because $t \ge |w-w^0|$ componentwise, both vectors are nonnegative. Moreover,

$$
w^+ - w^- = w - w^0,
\qquad
w^+ + w^- = t.
$$

Therefore,

$$
w = w^0 + w^+ - w^-,
\qquad
\sum_{i=1}^n (w_i^+ + w_i^-) = \sum_{i=1}^n t_i \le \tau.
$$

So every feasible 2-fold solution induces a feasible 3-fold solution.

The auxiliary variables are related by

$$
t = w^+ + w^-,
\qquad
w^+ = \frac{t + (w-w^0)}{2},
\qquad
w^- = \frac{t - (w-w^0)}{2}.
$$

Since the objective function depends only on $w$, both formulations produce the same feasible portfolios, the same optimal objective value, and the same optimal portfolio weights.

The only difference is dimensional: the 2-fold formulation works in $\mathbb{R}^{2n}$, whereas the 3-fold formulation works in $\mathbb{R}^{3n}$. Thus, the 2-fold method is a more compact but fully equivalent reformulation of the 3-fold approach.

In [9]:
def _linearize_turnover_constraint_2fold(
    self,
    x_init: np.ndarray,
    to_budget: float = float("inf"),
) -> None:
    """
    2-fold linearization with variables [w, t], where t_i >= |w_i - w0_i|.
    """
    x_init = np.asarray(x_init, dtype=float).reshape(-1)

    P = self.problem_data.get("P")
    q = self.problem_data.get("q")
    G = self.problem_data.get("G")
    h = self.problem_data.get("h")
    A = self.problem_data.get("A")
    b = self.problem_data.get("b")
    lb = self.problem_data.get("lb")
    ub = self.problem_data.get("ub")

    n = len(q)
    if len(x_init) != n:
        raise ValueError("x_init must have the same length as q.")

    # Objective
    P_new = np.zeros((2 * n, 2 * n))
    P_new[:n, :n] = P

    q_new = np.zeros(2 * n)
    q_new[:n] = q

    # Equalities: [A | 0]
    if A is None:
        A_new = None
        b_new = b
    else:
        a_rows = A.shape[0]
        A_new = np.zeros((a_rows, 2 * n))
        A_new[:, :n] = A
        b_new = b

    # Inequalities:
    # original:          [G | 0] [w, t] <= h
    # turnover upper:    [ I | -I] [w, t] <=  x_init
    # turnover lower:    [-I | -I] [w, t] <= -x_init
    # budget on t:       [ 0 | 1 ] [w, t] <= to_budget
    g_rows = 0 if G is None else G.shape[0]

    G_new = np.zeros((g_rows + 2 * n + 1, 2 * n))
    h_new = np.zeros(g_rows + 2 * n + 1)

    if G is not None:
        G_new[:g_rows, :n] = G
        h_new[:g_rows] = h

    # w - t <= x_init
    G_new[g_rows:g_rows + n, :n] = np.eye(n)
    G_new[g_rows:g_rows + n, n:] = -np.eye(n)
    h_new[g_rows:g_rows + n] = x_init

    # -w - t <= -x_init
    G_new[g_rows + n:g_rows + 2 * n, :n] = -np.eye(n)
    G_new[g_rows + n:g_rows + 2 * n, n:] = -np.eye(n)
    h_new[g_rows + n:g_rows + 2 * n] = -x_init

    # sum(t) <= to_budget
    G_new[-1, n:] = 1.0
    h_new[-1] = to_budget

    # Bounds
    lb_w = -np.inf * np.ones(n) if lb is None else np.asarray(lb, dtype=float)
    ub_w =  np.inf * np.ones(n) if ub is None else np.asarray(ub, dtype=float)

    lb_new = np.concatenate([lb_w, np.zeros(n)])
    ub_new = np.concatenate([ub_w, np.inf * np.ones(n)])

    self.update_problem_data({
        "P": P_new,
        "q": q_new,
        "G": G_new,
        "h": h_new,
        "A": A_new,
        "b": b_new,
        "lb": lb_new,
        "ub": ub_new,
    })


def _linearize_turnover_constraint_3fold(
    self,
    x_init: np.ndarray,
    to_budget: float = float("inf"),
) -> None:
    """
    3-fold linearization with variables [w, w_plus, w_minus],
    where w = x_init + w_plus - w_minus.
    """
    x_init = np.asarray(x_init, dtype=float).reshape(-1)

    P = self.problem_data.get("P")
    q = self.problem_data.get("q")
    G = self.problem_data.get("G")
    h = self.problem_data.get("h")
    A = self.problem_data.get("A")
    b = self.problem_data.get("b")
    lb = self.problem_data.get("lb")
    ub = self.problem_data.get("ub")

    n = len(q)
    if len(x_init) != n:
        raise ValueError("x_init must have the same length as q.")

    # Objective
    P_new = np.zeros((3 * n, 3 * n))
    P_new[:n, :n] = P

    q_new = np.zeros(3 * n)
    q_new[:n] = q

    # Equalities:
    # original: [A | 0 | 0] [w, w+, w-] = b
    # linking:  [I |-I | I] [w, w+, w-] = x_init
    a_rows = 0 if A is None else A.shape[0]
    A_new = np.zeros((a_rows + n, 3 * n))
    b_new = np.zeros(a_rows + n)

    if A is not None:
        A_new[:a_rows, :n] = A
        b_new[:a_rows] = b

    A_new[a_rows:, :n] = np.eye(n)
    A_new[a_rows:, n:2 * n] = -np.eye(n)
    A_new[a_rows:, 2 * n:] = np.eye(n)
    b_new[a_rows:] = x_init

    # Inequalities:
    # original turnover-independent inequalities
    # plus sum(w+) + sum(w-) <= to_budget
    g_rows = 0 if G is None else G.shape[0]
    G_new = np.zeros((g_rows + 1, 3 * n))
    h_new = np.zeros(g_rows + 1)

    if G is not None:
        G_new[:g_rows, :n] = G
        h_new[:g_rows] = h

    G_new[-1, n:2 * n] = 1.0
    G_new[-1, 2 * n:] = 1.0
    h_new[-1] = to_budget

    # Bounds
    lb_w = -np.inf * np.ones(n) if lb is None else np.asarray(lb, dtype=float)
    ub_w =  np.inf * np.ones(n) if ub is None else np.asarray(ub, dtype=float)

    lb_new = np.concatenate([lb_w, np.zeros(n), np.zeros(n)])
    ub_new = np.concatenate([ub_w, np.inf * np.ones(n), np.inf * np.ones(n)])

    self.update_problem_data({
        "P": P_new,
        "q": q_new,
        "G": G_new,
        "h": h_new,
        "A": A_new,
        "b": b_new,
        "lb": lb_new,
        "ub": ub_new,
    })


def linearize_turnover_constraint(
    self,
    x_init: np.ndarray,
    to_budget: float = float("inf"),
    method: str = "2fold",
) -> None:
    """
    Linearize the turnover constraint using either the 2-fold or 3-fold method.

    Parameters
    ----------
    x_init : np.ndarray
        Initial portfolio weights.
    to_budget : float
        Maximum allowed turnover.
    method : str
        One of: '2fold', '3fold', '2', '3'.
    """
    method_clean = str(method).lower().replace("-", "").replace("_", "")

    if method_clean in {"2", "2fold", "twofold"}:
        return self._linearize_turnover_constraint_2fold(
            x_init=x_init,
            to_budget=to_budget,
        )

    if method_clean in {"3", "3fold", "threefold"}:
        return self._linearize_turnover_constraint_3fold(
            x_init=x_init,
            to_budget=to_budget,
        )

    raise ValueError("method must be one of {'2fold', '3fold', '2', '3'}.")

## Demo

#### Create P and q

In [10]:
# Load the msci country index data
N = 10
data = load_data_msci(path = '../data/', n=N)
X = data['return_series']

# Compute the vector of expected returns (mean returns)
q = ExpectedReturn(method='geometric').estimate(X=X, inplace=False)

# Compute the covariance matrix
P = Covariance(method='pearson').estimate(X=X, inplace=False)

# q, P

### Create some constraints, instantiate an object of class QuadraticProgram, and add the method linearize_turnover_constraint to the instance.

In [11]:
# Instantiate the constraints with only the budget and long-only constraints
constraints = Constraints(ids = X.columns.tolist())
constraints.add_budget(rhs=1, sense='=')
constraints.add_box(lower=0.0, upper=1.0)
GhAb = constraints.to_GhAb()

In [12]:
# Create two independent quadratic programs (one for the 2-fold and one for the 3-fold linearization) with the same base data
qp_2f = QuadraticProgram(
    P=P.to_numpy(),
    q=q.to_numpy() * 0,
    G=GhAb['G'],
    h=GhAb['h'],
    A=GhAb['A'],
    b=GhAb['b'],
    lb=constraints.box['lower'].to_numpy(),
    ub=constraints.box['upper'].to_numpy(),
    solver='cvxopt',
)

qp_3f = QuadraticProgram(
    P=P.to_numpy(),
    q=q.to_numpy() * 0,
    G=GhAb['G'],
    h=GhAb['h'],
    A=GhAb['A'],
    b=GhAb['b'],
    lb=constraints.box['lower'].to_numpy(),
    ub=constraints.box['upper'].to_numpy(),
    solver='cvxopt',
)

# Bind methods to each instance
for obj in [qp_2f, qp_3f]:
    obj._linearize_turnover_constraint_2fold = types.MethodType(
        _linearize_turnover_constraint_2fold, obj
    )
    obj._linearize_turnover_constraint_3fold = types.MethodType(
        _linearize_turnover_constraint_3fold, obj
    )
    obj.linearize_turnover_constraint = types.MethodType(
        linearize_turnover_constraint, obj
    )

# Initialize the portfolio weights for the turnover constraint linearization
x_init_2f = pd.Series([1 / X.shape[1]] * X.shape[1], index=X.columns)
x_init_3f = x_init_2f.copy()

### Add a turnover limit of 50%. Solve the problem and check whether the turnover constraint is respected.

In [13]:
# Add the linearized turnover constraint
qp_2f.linearize_turnover_constraint(x_init=x_init_2f, to_budget=0.5, method='2fold')

# Check the updated problem data dimensions (should be (2N, 2N))
qp_2f.problem_data['P'].shape

# Solve the problem
qp_2f.solve()

# Check the turnover
solution = qp_2f.results.get('solution')
ids = constraints.ids
weights = pd.Series(solution.x[:len(ids)], index=ids)

print("Turnover (2-Fold Method):")
print(np.abs(weights - x_init_2f).sum())


Turnover (2-Fold Method):
0.49954552248142026


In [14]:
# Prepare initial weights
x_init_3f = pd.Series([1/X.shape[1]]*X.shape[1], index=X.columns)

# Add the linearized turnover constraint using 3-fold method
qp_3f.linearize_turnover_constraint(x_init=x_init_3f, to_budget=0.5, method='3fold')

# Check the updated problem data dimensions (should be (3N, 3N))
qp_3f.problem_data['P'].shape

# Solve the problem
qp_3f.solve()

# Check the turnover
solution = qp_3f.results.get('solution')
ids = constraints.ids
weights = pd.Series(solution.x[:len(ids)], index=ids)

print("Turnover (3-Fold Method):")
print(np.abs(weights - x_init_3f).sum())


Turnover (3-Fold Method):
0.49989601681867357


### Results and Interpretation

To validate the turnover linearization, the portfolio optimization problem was solved using both the **2-fold** and **3-fold** formulations with a turnover budget of

$$
\tau = 0.5.
$$

The resulting realized turnovers are:

- **2-fold method:**  
  $$
  \sum_{i=1}^n |w_i - w_i^0| = 0.49954552248142026
  $$

- **3-fold method:**  
  $$
  \sum_{i=1}^n |w_i - w_i^0| = 0.49989601681867357
  $$

Both values are strictly below the imposed turnover budget of \(0.5\), which confirms that the linearized constraints have been implemented correctly in both cases.

These results also illustrate the theoretical equivalence between the two formulations. Although the **2-fold method** introduces one auxiliary variable per asset and the **3-fold method** introduces separate buy and sell variables, both formulations generate the same feasible set in terms of portfolio weights \(w\). Therefore, both correctly enforce the turnover constraint

$$
\sum_{i=1}^{n} |w_i - w_i^0| \le \tau.
$$

The small numerical difference between the two realized turnover values is expected and is due to solver tolerances and numerical precision, not to a structural difference between the formulations.

In conclusion, the numerical results confirm that:

1. the turnover budget is respected in both formulations;
2. the 2-fold and 3-fold linearizations are both valid;
3. the 2-fold formulation provides a more compact implementation, while the 3-fold formulation gives a more explicit economic interpretation in terms of purchases and sales.